In [ ]:
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

siim_isic_melanoma_classification_path = kagglehub.competition_download('siim-isic-melanoma-classification')

print('Data source import complete.')


 74%|███████▍  | 78.6G/106G [17:41<06:06, 79.5MB/s]


OSError: [Errno 28] No space left on device

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<img src='https://i.imgur.com/odXiwBt.png'>
<h1><center>🧬 Melanoma Classification🧬: EDA + Augmentations</center><h1>

<img src='https://i.imgur.com/Jxtc8x0.png' width=500>

# 1. Introduction ▶

### 1.1 What is Melanoma? Stats and Facts:
* [Melanoma is the least common but the most deadly skin cancer, accounting for only about 1% of all cases, but the vast majority of skin cancer death.](https://www.aimatmelanoma.org/about-melanoma/melanoma-stats-facts-and-figures/)
* Melanoma is the third most common cancer among men and women ages 20-39.
* In the U.S., melanoma continues to be
    * the fifth most common cancer in men of all age groups
    * the sixth most common cancer in women of all age groups
* The world’s highest incidence of melanoma is in Australia and New Zealand (more than twice as high as in North America)


### 1.2 What we need to do? Data and Overview:
> The purpose is to correctly identify the **benign** and **malignant** cases. A *benign* tumor is a tumor that DOES NOT invade its surrounding tissue or spread around the body. A *malignant* tumor is a tumor that MAY invade its surrounding tissue or spread around the body.
<img src = 'https://www.verywellhealth.com/thmb/IFgBpbmhYCJdS4rvLACzX3Ukqsc=/1500x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/514240-article-img-malignant-vs-benign-tumor2111891f-54cc-47aa-8967-4cd5411fdb2f-5a2848f122fa3a0037c544be.png' width = 300>

> Data: DICOM Files split in Train (33,126 observations) and Test (10,982 observations)
<img src='https://i.imgur.com/or0AoVs.png' width = 500>

### 1.3 Metrics of Evaluation. Area under the ROC curve:
* [The ROC curve is created by plotting the true positive rate (TPR) against the false positive rate (FPR) at various threshold settings.](https://en.wikipedia.org/wiki/Receiver_operating_characteristic)

# 2. Libraries 📚

In [ ]:
# Regular Imports
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow info and warnings
import pandas as pd
import numpy as np
import shutil
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.image as mpimg
from tabulate import tabulate
import missingno as msno
from IPython.display import display_html
from PIL import Image
import gc
import cv2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# TensorFlow & Keras
from tensorflow.keras.applications import EfficientNetB0, VGG16
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, models  # Correctly importing layers and models

# For DICOM
import pydicom  # for DICOM images
from skimage.transform import resize

import warnings
warnings.filterwarnings("ignore")

# Set Color Palettes for the notebook
colors_nude = ['#e0798c','#65365a','#da8886','#cfc4c4','#dfd7ca']
sns.palplot(sns.color_palette(colors_nude))

# Set Style
sns.set_style("whitegrid")
sns.despine(left=True, bottom=True)


In [ ]:
list(os.listdir('../input/siim-isic-melanoma-classification'))

# 3. CSV Files - Train📁 + Test📂

In [ ]:
# Directory
directory = '../input/siim-isic-melanoma-classification'

# Import the 2 csv s
train_df = pd.read_csv(directory + '/train.csv')
test_df = pd.read_csv(directory + '/test.csv')

print('Train has {:,} rows and Test has {:,} rows.'.format(len(train_df), len(test_df)))

# Change columns names
new_names = ['dcm_name', 'ID', 'sex', 'age', 'anatomy', 'diagnosis', 'benign_malignant', 'target']
train_df.columns = new_names
test_df.columns = new_names[:5]

In [ ]:
train_df.head(5)

In [ ]:
test_df.head(5)

In [ ]:
df1_styler = train_df.head().style.set_table_attributes("style='display:inline'").set_caption('Head Train Data')
df2_styler = test_df.head().style.set_table_attributes("style='display:inline'").set_caption('Head Test Data')

display_html(df1_styler._repr_html_() + df2_styler._repr_html_(), raw=True)

## 3.1 Missing Values ❓

Let's first visualize the missing values.

In [ ]:
f, (ax1, ax2) = plt.subplots(1, 2, figsize = (16, 6))

msno.matrix(train_df, ax = ax1, color=(207/255, 196/255, 171/255), fontsize=10)
msno.matrix(test_df, ax = ax2, color=(218/255, 136/255, 130/255), fontsize=10)

ax1.set_title('Train Missing Values Map', fontsize = 16)
ax2.set_title('Test Missing Values Map', fontsize = 16);

**Train**:
1. `sex`: 65 missing values (0.2% of total data)
2. `age`: 68 missing values (correspond with `sex` missingness)
3. `anatomy`: 527 missing values (1.59% of total data)

**Test**:
1. `anatomy`: 351 missing values (3.1% of total data)

Let's take them 1 by 1 and deal with em.

### Train: SEX Variable

> All missing values are *benign* and the majority of the patients have the Melanoma in the Lower Extremity, Upper Extremity and Torso. All values for `diagnosis` are unknown. Therefore, we'll use the most predominant gender that appears in these values to impute the missing values.

In [ ]:
# Separate Data
nan_sex = train_df[train_df['sex'].isna()]
is_sex = train_df[~train_df['sex'].isna()]

# Plotting
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(data=nan_sex, x='anatomy', ax=ax1, palette=colors_nude)
sns.countplot(data=is_sex, x='anatomy', ax=ax2, palette=colors_nude)

ax1.set_title('NAN Gender: Anatomy', fontsize=16)
ax2.set_title('Rest Gender: Anatomy', fontsize=16)

for ax in [ax1, ax2]:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
sns.despine(left=True, bottom=True)

# Benign/Malignant Check
benign_count = nan_sex['benign_malignant'].value_counts().get('benign', 0)
malignant_count = nan_sex['benign_malignant'].value_counts().get('malignant', 0)
total_nan = nan_sex.shape[0]

print(f'Out of {total_nan} NAN values, {benign_count} are benign and {malignant_count} are malignant.')


In [ ]:
# Check how many are males and how many females
anatomy = ['lower extremity', 'upper extremity', 'torso']
train_df[(train_df['anatomy'].isin(anatomy)) & (train_df['target'] == 0)]['sex'].value_counts()

In [ ]:
# Impute the missing values with male
train_df['sex'].fillna("male", inplace = True)

### Train: AGE Variable
> The distributions and values are very similar with the missingness patern in `sex` variable. So, we'll impute in the same manner. The *mean* and *median* of `age` variable has the same value of 50, while the *mode* is at 45. The distribution is normal, so we'll use the MEDIAN to impute.

In [ ]:
# Separate Data
nan_age = train_df[train_df['age'].isna()]
is_age = train_df[~train_df['age'].isna()]

# Plotting
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(data=nan_age, x='anatomy', ax=ax1, palette=colors_nude)
sns.countplot(data=is_age, x='anatomy', ax=ax2, palette=colors_nude)

ax1.set_title('NAN Age: Anatomy', fontsize=16)
ax2.set_title('Rest Age: Anatomy', fontsize=16)

for ax in [ax1, ax2]:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
sns.despine(left=True, bottom=True)

# Benign/Malignant Check
benign_count = nan_age['benign_malignant'].value_counts().get('benign', 0)
malignant_count = nan_age['benign_malignant'].value_counts().get('malignant', 0)
total_nan = nan_age.shape[0]

print(f'Out of {total_nan} NAN values, {benign_count} are benign and {malignant_count} are malignant.')


In [ ]:
# Check the mode age
anatomy = ['lower extremity', 'upper extremity', 'torso']
mode = train_df[(train_df['anatomy'].isin(anatomy)) & (train_df['target'] == 0) & (train_df['sex'] == 'male')]['age'].mode()
print('Mode is:', mode)

In [ ]:
# Impute the missing values with Mode
train_df['age'].fillna(mode, inplace = True)

### Train: ANATOMY Variable
> First, we need to keep in mind that between the missing data there are 9 malignant cases, so we should treat the imputation separate for both benign and malignant. In terms of `age` and `gender`, both missing and not missing data seem to behave about the same. However, the most frequent anatomy for both benign and malignant is *torso*, so we'll impute this value.

In [ ]:
# Impute the missing values with Mode
train_df['age'].fillna(mode, inplace = True)

In [ ]:
# Impute for anatomy
train_df['anatomy'].fillna('torso', inplace = True)

### Test: ANATOMY Variable
> The majority of the people with missing `anatomy` have 70 yo, so we'll use the anatomy with the biggest frequency for age 70.

In [ ]:
# Add flag for missing anatomy
anatomy = test_df.copy()
anatomy['flag'] = np.where(anatomy['anatomy'].isna(), 'missing', 'not_missing')

# Figure
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Corrected countplot
sns.countplot(x='flag', hue='sex', data=anatomy, ax=ax1, palette=colors_nude)
ax1.set_title('Gender for Anatomy', fontsize=16)

# Replace sns.distplot with sns.kdeplot (since distplot is deprecated)
sns.kdeplot(data=anatomy[anatomy['flag'] == 'missing'], x='age',
            label='Missing', ax=ax2, color=colors_nude[2], linewidth=4, bw_adjust=0.1)
sns.kdeplot(data=anatomy[anatomy['flag'] == 'not_missing'], x='age',
            label='Not Missing', ax=ax2, color=colors_nude[3], linewidth=4, bw_adjust=0.1)
ax2.set_title('Age Distribution for Anatomy', fontsize=16)
ax2.legend(title='Flag')

sns.despine(left=True, bottom=True)


In [ ]:
anatomy_counts = test_df[test_df['age'] == 70]['anatomy'].value_counts().reset_index()
anatomy_counts

In [ ]:
# Impute the value
test_df['anatomy'].fillna('torso', inplace = True)

In [ ]:
# Save the files
train_df.to_csv('train_clean.csv', index=False)
test_df.to_csv('test_clean.csv', index=False)

## 3.2 EDA - Let's take a look 🔎

### Target Variable:
1. Very HIGH class imbalance. We need to take this in consideration when Modeling.
2. Age distribution:
    * Benign: follows a normal distribution
    * Malignant: a little skewed to the left, with the peak oriented towards higher age values.

In [ ]:
# Figure setup
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Count plot for benign and malignant cases
sns.countplot(data=train_df, x='benign_malignant', palette=colors_nude[2:4], ax=ax1)
for p in ax1.patches:
    ax1.annotate(
        format(p.get_height(), ','),
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='center', xytext=(0, 4), textcoords='offset points'
    )
ax1.set_title('Benign vs Malignant Cases', fontsize=14)

# Age distribution for benign and malignant cases
sns.kdeplot(data=train_df[train_df['target'] == 0], x='age', ax=ax2, color=colors_nude[2],
            linewidth=3, label='Benign')
sns.kdeplot(data=train_df[train_df['target'] == 1], x='age', ax=ax2, color=colors_nude[3],
            linewidth=3, label='Malignant')
ax2.set_title('Age Distribution by Target Type', fontsize=14)
ax2.legend()

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()


### Target and Genders:
1. There are more males than females in the dataset
2. However, the percentages are ~ the same

In [ ]:
plt.figure(figsize=(16, 6))
a = sns.countplot(data=train_df, x='benign_malignant', hue='sex', palette=colors_nude)

for p in a.patches:
    a.annotate(format(p.get_height(), ','),
           (p.get_x() + p.get_width() / 2.,
            p.get_height()), ha = 'center', va = 'center',
           xytext = (0, 4), textcoords = 'offset points')

plt.title('Gender split by Target Variable', fontsize=16)
sns.despine(left=True, bottom=True);

### Anatomy and Target
> Note: Distributions are about the same shape for both benign and malignant cases.

In [ ]:
plt.figure(figsize=(16, 6))
a = sns.countplot(data=train_df, x='benign_malignant', hue='anatomy', palette=colors_nude)

for p in a.patches:
    a.annotate(format(p.get_height(), ','),
           (p.get_x() + p.get_width() / 2.,
            p.get_height()), ha = 'center', va = 'center',
           xytext = (0, 4), textcoords = 'offset points')

plt.title('Anatomy split by Target Variable', fontsize=16)
sns.despine(left=True, bottom=True);

### Diagnosis and Target

In [ ]:
# Filter data to exclude 'unknown'
filtered_df = train_df[train_df['diagnosis'] != 'unknown']

# Create plots
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot benign cases
a = sns.countplot(data=filtered_df[filtered_df['target'] == 0], x='diagnosis', ax=ax1, palette=colors_nude)

# Plot malignant cases
b = sns.countplot(data=filtered_df[filtered_df['target'] == 1], x='diagnosis', ax=ax2, palette=colors_nude)

# Rotate x-axis labels for better visibility
a.set_xticklabels(a.get_xticklabels(), rotation=35, ha="right")
b.set_xticklabels(b.get_xticklabels(), rotation=35, ha="right")

# Annotate bar heights
for p in a.patches:
    a.annotate(format(p.get_height(), ','),
               (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='center', xytext=(0, 4), textcoords='offset points')

for p in b.patches:
    b.annotate(format(p.get_height(), ','),
               (p.get_x() + p.get_width() / 2., p.get_height()),
               ha='center', va='center', xytext=(0, 4), textcoords='offset points')

# Set titles
ax1.set_title('Benign Cases: Diagnosis View', fontsize=16)
ax2.set_title('Malignant Cases: Diagnosis View', fontsize=16)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()


### Test Dataset Overview
> Distributions look ~ the same as in Train Data.

In [ ]:
# Calculate the counts for each gender
sex_counts = train_df['sex'].value_counts()

# Define sizes and labels
size = sex_counts.values
labels = sex_counts.index
explode = [0.1 if label == 'male' else 0 for label in labels]  # Explode male for emphasis (optional)

# Plot
plt.figure(figsize=(5, 5))
plt.pie(size, labels=labels, explode=explode, shadow=True, startangle=90, colors=["r", "c"], autopct='%1.1f%%')
plt.title("Gender Distribution (Male vs Female)", fontsize=18)
plt.legend(title="Gender")
plt.show()

# Print counts
for gender, count in zip(labels, size):
    print(f"{gender.capitalize()} Cases = {count}")


In [ ]:
# Calculate the counts for each anatomy
anatomy_counts = train_df['anatomy'].value_counts()

# Define labels and sizes
size = anatomy_counts.values
labels = anatomy_counts.index

# Plotting bar chart
plt.figure(figsize=(7, 5))
plt.bar(labels, size, color=["r", "c"])

# Title and labels
plt.title("Anatomy Distribution", fontsize=18)
plt.xlabel("Anatomy", fontsize=14)
plt.ylabel("Count", fontsize=14)

# Rotate x-axis labels
plt.xticks(rotation=45)  # You can change the angle to any value you prefer

# Print counts
for anatomy, count in zip(labels, size):
    print(f"{anatomy.capitalize()} Cases = {count}")

plt.show()


# 4. Preprocess .csv files 📐

## 4.1 Add Image Path
This will help access the images in the feature.

In [ ]:
# Step 1: Filter train.csv
malignant = train_df[train_df['target'] == 1]
benign = train_df[train_df['target'] == 0]

required_benign = 15000 - len(malignant)
benign = benign.sample(n=required_benign, random_state=42)

train_reduced = pd.concat([malignant, benign])
print(f"Reduced Train Size: {len(train_reduced)}")

# Step 2: Filter test.csv
test_reduced = test_df.sample(n=8000, random_state=42)
print(f"Reduced Test Size: {len(test_reduced)}")

# Step 3: Copy Required Images
def manage_images(src_folder, dst_folder, valid_image_names):
    os.makedirs(dst_folder, exist_ok=True)
    valid_image_paths = set(valid_image_names + ".jpg")  # Convert image names to full file names
    for img_file in os.listdir(src_folder):
        src_path = os.path.join(src_folder, img_file)
        dst_path = os.path.join(dst_folder, img_file)
        if img_file in valid_image_paths:
            shutil.copy(src_path, dst_path)

# Copy train images
manage_images(
    src_folder="/kaggle/input/siim-isic-melanoma-classification/jpeg/train",
    dst_folder="/kaggle/working/train",
    valid_image_names=train_reduced['dcm_name'].values  # Pass the column as a list
)

# Copy test images
manage_images(
    src_folder="/kaggle/input/siim-isic-melanoma-classification/jpeg/test",
    dst_folder="/kaggle/working/test",
    valid_image_names=test_reduced['dcm_name'].values  # Pass the column as a list
)

# Save reduced CSV files
train_reduced.to_csv("/kaggle/working/train_reduced.csv", index=False)
test_reduced.to_csv("/kaggle/working/test_reduced.csv", index=False)

print("Filtered data and images have been saved successfully!")


In [ ]:
# Verify the number of images in the reduced train and test datasets
train_images = os.listdir("/kaggle/working/train")
test_images = os.listdir("/kaggle/working/test")

# Print counts
print(f"Number of train images after reduction: {len(train_images)}")
print(f"Number of test images after reduction: {len(test_images)}")

# Display a few train image samples
print("\nSample train images:")
print(train_images[:5])

# Display a few test image samples
print("\nSample test images:")
print(test_images[:5])


In [ ]:
# # # === DICOM ===
# # # Create the paths
# # path_train = directory + '/train/' + train_df['dcm_name'] + '.dcm'
# # path_test = directory + '/test/' + test_df['dcm_name'] + '.dcm'

# # Append to the original dataframes
# # train_df['path_dicom'] = path_train
# # test_df['path_dicom'] = path_test

# # === JPEG ===
# # Create the paths
# path_train = directory + '/jpeg/train/' + train_df['dcm_name'] + '.jpg'
# path_test = directory + '/jpeg/test/' + test_df['dcm_name'] + '.jpg'

# # # Append to the original dataframes
# train_df['path_jpeg'] = path_train
# test_df['path_jpeg'] = path_test

In [ ]:
# Create a new folder for reduced images
reduced_image_folder = "/path/to/reduced_images"
os.makedirs(reduced_image_folder, exist_ok=True)

# Assuming `train_reduced` and `test_reduced` are the reduced dataframes with the 'image_name' column

# Path to the original train image folder
original_train_folder = "/kaggle/input/siim-isic-melanoma-classification/jpeg/train"
original_test_folder = "/kaggle/input/siim-isic-melanoma-classification/jpeg/test"

# Function to copy images to the reduced image folder and update paths in the dataframe
def update_image_paths_and_copy_images(data, original_folder, reduced_folder):
    updated_paths = []

    # Loop through each image in the reduced dataset
    for image_name in data['dcm_name']:
        original_image_path = os.path.join(original_folder, f"{image_name}.jpg")
        new_image_path = os.path.join(reduced_folder, f"{image_name}.jpg")

        # Copy the image to the new folder
        shutil.copy(original_image_path, new_image_path)

        # Update the path in the list
        updated_paths.append(new_image_path)

    # Add the updated paths to the dataframe
    data['image_path'] = updated_paths
    return data

# Update train and test data with new image paths
train_reduced_updated = update_image_paths_and_copy_images(train_reduced, original_train_folder, reduced_image_folder)
test_reduced_updated = update_image_paths_and_copy_images(test_reduced, original_test_folder, reduced_image_folder)

# Save the updated data to new CSV files with updated image paths
train_reduced_updated.to_csv("train_reduced_updated.csv", index=False)
test_reduced_updated.to_csv("test_reduced_updated.csv", index=False)

print("Updated CSV files saved as train_reduced_updated.csv and test_reduced_updated.csv")


In [ ]:
# Create new folders for reduced DICOM files
reduced_dicom_train_folder = "/path/to/reduced_dicom_train"
reduced_dicom_test_folder = "/path/to/reduced_dicom_test"

# Create the folders if they do not exist
os.makedirs(reduced_dicom_train_folder, exist_ok=True)
os.makedirs(reduced_dicom_test_folder, exist_ok=True)

# Assuming `train_reduced` and `test_reduced` are the reduced dataframes with the 'dcm_name' column

# Path to the original train/test DICOM folder
original_dicom_train_folder = "/kaggle/input/siim-isic-melanoma-classification/train"
original_dicom_test_folder = '/kaggle/input/siim-isic-melanoma-classification/test'

# Function to copy DICOM files to the reduced folder and update paths in the dataframe
def update_dicom_paths_and_copy_files(data, original_dicom_folder, reduced_dicom_folder):
    updated_dicom_paths = []

    # Loop through each DICOM file in the reduced dataset
    for dcm_name in data['dcm_name']:
        original_dcm_path = os.path.join(original_dicom_folder, f"{dcm_name}.dcm")
        new_dcm_path = os.path.join(reduced_dicom_folder, f"{dcm_name}.dcm")

        # Copy the DICOM file to the new folder
        shutil.copy(original_dcm_path, new_dcm_path)

        # Update the path in the list
        updated_dicom_paths.append(new_dcm_path)

    # Add the updated DICOM paths to the dataframe
    data['dcm_path'] = updated_dicom_paths
    return data

# Update train and test data with new DICOM paths
train_reduced_updated_dcm = update_dicom_paths_and_copy_files(train_reduced, original_dicom_train_folder, reduced_dicom_train_folder)
test_reduced_updated_dcm = update_dicom_paths_and_copy_files(test_reduced, original_dicom_test_folder, reduced_dicom_test_folder)

# Save the updated data to new CSV files with updated DICOM paths
train_reduced_updated_dcm.to_csv("train_reduced_updated_dcm.csv", index=False)
test_reduced_updated_dcm.to_csv("test_reduced_updated_dcm.csv", index=False)

print("Updated CSV files with DICOM paths saved as 'train_reduced_updated_dcm.csv' and 'test_reduced_updated_dcm.csv'")


In [ ]:
df_train  = train_reduced_updated
df_test  = test_reduced_updated

df_train.to_csv("df_train.csv", index=False)
df_test.to_csv("df_train.csv", index=False)

## 4.2 One Hot Encoding
Transforming all categorical features un numerical.
> Note1: `sex`, `anatomy`, `diagnosis` need to be encoded.

> Note2: `benign_malignant` column will be dropped, as the information is already in the `target` column.

# Train

In [ ]:
to_encode = ['sex', 'anatomy', 'diagnosis']
encoded_all = []

label_encoder = LabelEncoder()

for column in to_encode:
    encoded = label_encoder.fit_transform(df_train[column])
    encoded_all.append(encoded)

df_train['sex'] = encoded_all[0]
df_train['anatomy'] = encoded_all[1]
df_train['diagnosis'] = encoded_all[2]

if 'benign_malignant' in train_reduced.columns : train_reduced.drop(['benign_malignant'], axis=1, inplace=True)

# test

In [ ]:
to_encode = ['sex', 'anatomy']
encoded_all = []

label_encoder = LabelEncoder()

for column in to_encode:
    encoded = label_encoder.fit_transform(df_test[column])
    encoded_all.append(encoded)

df_test['sex'] = encoded_all[0]
df_test['anatomy'] = encoded_all[1]

 Save the files before continuing.

In [ ]:
# # Save the files
# train_df.to_csv('train_clean.csv', index=False)
# test_reduced.to_csv('test_clean.csv', index=False)

# # Display the contents of the train_df DataFrame
# # train_df
# #

In [ ]:
# train_clean = train_df  # Define train_clean if you need it
# train_clean.to_csv('train_clean.csv', index=False)


In [ ]:
# import os

# # Assuming data_dir is already defined
# train_reduced['image_path'] = train_reduced['dcm_name'].apply(lambda x: os.path.join(directory, 'jpeg/train', f'{x}.jpg'))


# 5. The Images 📸

There are 2 types of images containing the same information:
1. `.dcm` files: [DICOM files](https://en.wikipedia.org/wiki/DICOM). It's saved in the "Digital Imaging and Communications in Medicine" format. It contains an image from a medical scan, such as an ultrasound or MRI + information about the patient.
2. `.jpeg` files: the DICOM files converted into .jpeg format
3. `.tfrec` files: [The TFRecord file format is a simple record-oriented binary format for ML training data.](https://docs.databricks.com/applications/deep-learning/data-prep/tfrecords-to-tensorflow.html#:~:text=The%20TFRecord%20file%20format%20is,part%20of%20an%20input%20pipeline.)

## 5.1 Sanity Check
> Check if images in `.dcm` and `.jpeg` format have the same number of observations as in `train_df` and `test_df`.

In [ ]:
# print('Train .dcm number of images:', len(list(os.listdir('../input/siim-isic-melanoma-classification/train'))), '\n' +
#       'Test .dcm number of images:', len(list(os.listdir('../input/siim-isic-melanoma-classification/test'))), '\n' +
#       'Train .jpeg number of images:', len(list(os.listdir('../input/siim-isic-melanoma-classification/jpeg/train'))), '\n' +
#       'Test .jpeg number of images:', len(list(os.listdir('../input/siim-isic-melanoma-classification/jpeg/test'))), '\n' +
#       '-----------------------', '\n' +
#       'There is the same number of images as in train/ test .csv datasets')

## 5.2 DICOM Images

### Malignant vs Benign Images

Let's look at the difference between *malignant* and *benign* melanomas.

In [ ]:
def show_images(data, n=5, rows=1, cols=5, title='Default'):
    plt.figure(figsize=(16, 4))

    for k, path in enumerate(df_train['dcm_path'][:n]):
        image = pydicom.dcmread(path)  # Use dcmread to read the DICOM file
        image = image.pixel_array  # Extract pixel data

        # Optionally resize the image if needed
        # image = resize(image, (200, 200), anti_aliasing=True)

        plt.suptitle(title, fontsize=16)
        plt.subplot(rows, cols, k + 1)
        plt.imshow(image, cmap='gray')  # Use gray colormap for medical images
        plt.axis('off')


In [ ]:
# Show Benign Samples
show_images(df_train[df_train['target'] == 0], n=10, rows=2, cols=5, title='Benign Sample')

In [ ]:
# Show Malignant Samples
show_images(df_train[df_train['target'] == 1], n=10, rows=2, cols=5, title='Malignant Sample')

# 6. Class Imbalance ⚖

This is a **very** important topic in this classification problem, as the 2 classes we are dealing with are highly imbalanced, with 98% of the data being *benign* and only 2% of the data being *malignant*.

<img src='https://i.imgur.com/Oc4Z3EP.png' width=400>

This is also the kind of problem where you **DON'T** want to have False Negatives. It's waaayyy worse to tell a patient they don't have cancer when they actually do, than to tell em they do have it and they actually don't. So, having balanced classes is *crucial*.

### We can do:
* **Oversampling**: of the minority class, increasing the number of images through augmentations
* **Understampling**: of the majority class (we shall see how the process is going)

> <img src='https://i.imgur.com/OwvqMbQ.png' width=450>

### What is Data Augmentation?

Is *moving, rotating, cropping, flipping, changing color/brightness/hue and whatever else you can come up with* to change the aspect of the original image. It is helpful in Overfitting, as the model learns not only 1 aspect of the image, but multiple (a cat can be standing up straight, or funny upside down, in a b&w image etc.).
> A child recognizes a cat in all these random contexts ... but *your* model can? 😉

<img src="https://miro.medium.com/max/850/1*ae1tW5ngf1zhPRyh7aaM1Q.png" width = 450>

### Other things to keep in mind:
<div class="alert alert-block alert-info">
<p><b>#1:</b> Different skin tones. Might need to find something that levels that.</p>
<p><b>#2:</b> Different lightings in the image.</p>
<p><b>#3:</b> Different sizes of the images. We need todiv>

## 6.1 B&W View 🤍🖤

In [ ]:
# class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
# class_weights = dict(enumerate(class_weights))
# #

In [ ]:
# X_train_img, X_val_img, X_train_meta, X_val_meta, y_train, y_val = train_test_split(
#     X_train_img, metadata, y_train, test_size=0.2, random_state=42
# )


In [ ]:
# # Pre-trained EfficientNetB0
# base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
# base_model.trainable = False  # Freeze base model

# # Image input
# image_input = base_model.input
# x = base_model(image_input, training=False)
# x = GlobalAveragePooling2D()(x)  # Pooling

# # Metadata input
# meta_input = Input(shape=(X_train_meta.shape[1],))
# y = Dense(32, activation='relu')(meta_input)

# # Combine features
# combined = concatenate([x, y])
# z = Dense(64, activation='relu')(combined)
# z = Dropout(0.5)(z)
# output = Dense(1, activation='sigmoid')(z)  # Binary classification

# # Create model
# model = Model(inputs=[image_input, meta_input], outputs=output)
# model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# # Preprocess images and extract the target values
# def preprocess_images(image_paths, target_size=(128, 128)):
#     images = []
#     for path in image_paths:
#         img = load_img(path, target_size=target_size)  # Resize image to target size
#         img = img_to_array(img) / 255.0  # Normalize the image
#         images.append(img)
#     return np.array(images)

In [ ]:
# # Extract image paths and preprocess them
# X = preprocess_images(df_train['image_path'].values)
# y = df_train['target'].values  # Use target column for labels

In [ ]:
# # Split the data into training and validation sets
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# # Load pretrained model (e.g., VGG16)
# base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# # Freeze the base model layers
# base_model.trainable = False


In [ ]:
# # Build the model
# model = Sequential([
#     base_model,
#     GlobalAveragePooling2D(),  # Global pooling layer
#     Dropout(0.2),  # Dropout to prevent overfitting
#     Dense(128, activation='relu'),  # Fully connected layer
#     Dense(1, activation='sigmoid')  # Output layer (binary classification)
# ])

In [ ]:
# # Compile the model
# model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# # Data augmentation setup
# train_datagen = ImageDataGenerator(
#     rescale=1./255,
#     rotation_range=20,
#     width_shift_range=0.1,
#     height_shift_range=0.1,
#     shear_range=0.1,
#     zoom_range=0.1,
#     fill_mode='nearest'
# )

In [ ]:
# # Create data generators for training and validation
# train_generator = train_datagen.flow(X_train, y_train, batch_size=32)
# val_generator = ImageDataGenerator(rescale=1./255).flow(X_val, y_val, batch_size=32)

In [ ]:
# # Train the model
# history = model.fit(
#     train_generator,
#     validation_data=val_generator,
#     epochs=10,
#     steps_per_epoch=len(X_train) // 32,
#     validation_steps=len(X_val) // 32
# )

# # Save the model
# model.save('melanoma_classifier.h5')


# Split the data

In [ ]:
# Split the data into training and validation (80% train, 20% validation)
train_df, val_df = train_test_split(df_train, test_size=0.2, random_state=42)

# training and validation images

In [ ]:
# Preprocess the images (resize and normalize)
def preprocess_images(image_paths, target_size=(224, 224)):
    images = []
    for path in image_paths:
        img = load_img(path, target_size=target_size)
        img_array = img_to_array(img)
        img_array = img_array / 255.0  # Normalize pixel values
        images.append(img_array)
    return np.array(images)

In [ ]:
# Preprocess the training and validation images
X_train = preprocess_images(train_df['image_path'].values)
X_val = preprocess_images(val_df['image_path'].values)

# Get the labels
y_train = train_df['target'].values
y_val = val_df['target'].values

# EfficientNetB0 model

In [ ]:
# Build the model using EfficientNetB0
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the base model layers

# Create the model on top of EfficientNetB0
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Sigmoid for binary classification
])

In [ ]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()


# ImageDataGenerator

In [ ]:
# Set up ImageDataGenerator for data augmentation on training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Set up validation data (no augmentation, only rescaling)
val_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# Fit the model using the training and validation data
history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    epochs=10,
    validation_data=val_datagen.flow(X_val, y_val, batch_size=32)
)

# Evaluate the model

In [ ]:
# Evaluate the model on the validation set
val_loss, val_acc = model.evaluate(val_datagen.flow(X_val, y_val, batch_size=32))
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_acc}")

#  Save the model


In [ ]:
# Save the model
model.save('skin_cancer_model.h5')